# 07 · Ablation Grid

Aggregates the ablation ladder for `docs/BENCHMARKS.md` from:
- `runs/final_eval/comprehensive_evaluation.json` (baseline / fusion configurations)
- `runs/final_eval/calibration.json` (post-hoc calibration)
- `runs/final_eval/conformal.json` (coverage + CRC)

Regenerate all: `python scripts/regenerate_all_numbers.py`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

def _load(name):
    p = ROOT / 'runs' / 'final_eval' / name
    return json.loads(p.read_text()) if p.exists() else None

baseline = _load('comprehensive_evaluation.json')
cal      = _load('calibration.json')
conf     = _load('conformal.json')
shadow   = _load('shadow_threshold_tuning.json')
print('baseline:', 'ok' if baseline else 'missing')
print('calibration:', 'ok' if cal else 'missing')
print('conformal:', 'ok' if conf else 'missing')
print('shadow_tuning:', 'ok' if shadow else 'missing')

In [ ]:
rows = []
if baseline:
    for k, v in (baseline.get('results') or {}).items():
        rows.append({'stage': k, 'metric_value': v, 'metric_name': baseline.get('metric')})

if cal:
    for det in ('yolo', 'rfdetr', 'fusion'):
        d = cal.get(det, {})
        for stage in ('before', 'after'):
            s = d.get(stage)
            if not s: continue
            rows.append({
                'stage': f'{det}_{stage}',
                'metric_value': s.get('ece'),
                'metric_name': 'ECE',
                'brier': s.get('brier'),
                'nll': s.get('nll'),
            })

if conf:
    rows.append({
        'stage': 'conformal_alpha_0.05',
        'metric_value': conf['split_conformal']['empirical_coverage_test'],
        'metric_name': 'empirical_coverage',
        'target': conf['split_conformal']['target_coverage'],
    })

if shadow:
    rows.append({
        'stage': 'shadow_best_threshold',
        'metric_value': shadow.get('best_metric'),
        'metric_name': shadow.get('metric'),
        'threshold': shadow.get('best_threshold'),
    })

df = pd.DataFrame(rows)
df

In [ ]:
# Paste-ready Markdown for docs/BENCHMARKS.md
print(df.to_markdown(index=False, floatfmt='.4f'))